# PyFock vs. PySCF with the Skala functional

Both codes run the same neural functional, `skala-1.1`, so this compares the DFT engine around it.
PyFock reads the TorchScript checkpoint directly; the official `skala.pyscf` API drives PySCF.

Needs PySCF, so run this on Linux or macOS.

In [ ]:
%pip install -q "pyfock[skala,ase]" skala

In [ ]:
import os

NCORES = 4
os.environ['OMP_NUM_THREADS'] = str(NCORES)
os.environ['OPENBLAS_NUM_THREADS'] = str(NCORES)
os.environ['MKL_NUM_THREADS'] = str(NCORES)
os.environ['NUMEXPR_NUM_THREADS'] = str(NCORES)

import time
import numpy as np
from pyfock import Basis, DFT, DFT_Grad, Grids, Mol
from pyscf import gto, lib
from skala.pyscf import SkalaKS

lib.num_threads(NCORES)

## Warm-up

PyFock compiles its kernels with Numba on first use, and the AO-Hessian kernel the gradient needs is
never touched by the SCF. Run a small calculation first and throw it away, or the benzene timing below
measures the compiler.

In [ ]:
mol = Mol(atoms=[['O', 0.0, 0.0, 0.1173], ['H', 0.0, 0.7572, -0.4692], ['H', 0.0, -0.7572, -0.4692]])
basis = Basis(mol, {'all': Basis.load(mol=mol, basis_name='def2-SVP')})
auxbasis = Basis(mol, {'all': Basis.load(mol=mol, basis_name='def2-universal-jkfit')})

dft = DFT(mol, basis, auxbasis, xc='skala-1.1', grids=Grids(mol, level=3, verbose=False),
          dispersion=True)
dft.sao = True          # spherical AOs, to match PySCF
dft.conv_crit = 1e-8
dft.ncores = NCORES
dft.save_ao_values = True

dft.scf()
DFT_Grad(dft, verbose=False).calculate()
print('warm-up done')

## Benzene: SCF energy and forces

Same basis, auxiliary basis, grid level, convergence threshold and core count on both sides. Dispersion
is on in both (`SkalaKS` defaults to `with_dftd3=True`).

In [ ]:
benzene = [
    ['C',  0.0000,  1.3970, 0.0], ['C',  1.2098,  0.6985, 0.0], ['C',  1.2098, -0.6985, 0.0],
    ['C',  0.0000, -1.3970, 0.0], ['C', -1.2098, -0.6985, 0.0], ['C', -1.2098,  0.6985, 0.0],
    ['H',  0.0000,  2.4810, 0.0], ['H',  2.1486,  1.2405, 0.0], ['H',  2.1486, -1.2405, 0.0],
    ['H',  0.0000, -2.4810, 0.0], ['H', -2.1486, -1.2405, 0.0], ['H', -2.1486,  1.2405, 0.0],
]

In [ ]:
# --- PyFock
mol = Mol(atoms=[list(a) for a in benzene])
basis = Basis(mol, {'all': Basis.load(mol=mol, basis_name='def2-SVP')})
auxbasis = Basis(mol, {'all': Basis.load(mol=mol, basis_name='def2-universal-jkfit')})

dft = DFT(mol, basis, auxbasis, xc='skala-1.1', grids=Grids(mol, level=3, verbose=False),
          dispersion=True)
dft.sao = True
dft.conv_crit = 1e-8
dft.ncores = NCORES
dft.save_ao_values = True    # caches the AO values on the grid: ~16% faster SCF, ~0.4 GB for benzene

t0 = time.perf_counter()
energy_pyfock, _ = dft.scf()
scf_pyfock = time.perf_counter() - t0

t0 = time.perf_counter()
forces_pyfock = DFT_Grad(dft, verbose=False).calculate()['gradient']
grad_pyfock = time.perf_counter() - t0

print('PyFock  E = %.8f Ha   SCF %.1f s   forces %.1f s' % (energy_pyfock, scf_pyfock, grad_pyfock))

In [ ]:
# --- PySCF
mol = gto.M(atom=[[a[0], tuple(a[1:])] for a in benzene], basis='def2-SVP', verbose=0)

ks = SkalaKS(mol, xc='skala-1.1', with_density_fit=True, auxbasis='def2-universal-jkfit')
ks.grids.level = 3
ks.conv_tol = 1e-8

t0 = time.perf_counter()
energy_pyscf = ks.kernel()
scf_pyscf = time.perf_counter() - t0

t0 = time.perf_counter()
forces_pyscf = ks.nuc_grad_method().kernel()
grad_pyscf = time.perf_counter() - t0

print('PySCF   E = %.8f Ha   SCF %.1f s   forces %.1f s' % (energy_pyscf, scf_pyscf, grad_pyscf))

In [ ]:
print('SCF     PyFock %6.1f s   PySCF %6.1f s   speed-up %.2fx' % (scf_pyfock, scf_pyscf, scf_pyscf / scf_pyfock))
print('forces  PyFock %6.1f s   PySCF %6.1f s   speed-up %.2fx' % (grad_pyfock, grad_pyscf, grad_pyscf / grad_pyfock))
print()
print('energy difference     %.2e Ha' % abs(energy_pyfock - energy_pyscf))
print('max force difference  %.2e Ha/Bohr' % np.abs(forces_pyfock - forces_pyscf).max())

The functional is identical, so the energies differ only through the engine: the grids are not
point-for-point the same and the Coulomb term is density-fitted. Agreement around 1e-6 Ha is what to
expect, not bitwise equality.

## Geometry optimization

Water and ethane, same ASE optimizer and same `fmax` on both sides. Only the DFT engine differs, so the
final bond lengths should agree.

In [ ]:
from ase import Atoms
from ase.optimize import LBFGSLineSearch
from pyfock import PyFockCalculator
from skala.ase import Skala

water_start = Atoms('OHH', positions=[[0.0, 0.0, 0.12], [0.0, 0.80, -0.48], [0.0, -0.80, -0.48]])
ethane_start = Atoms('C2H6', positions=[
    [ 0.0000,  0.0000,  0.7650], [ 0.0000,  0.0000, -0.7650],
    [ 0.0000,  1.0180,  1.1630], [ 0.8817, -0.5090,  1.1630], [-0.8817, -0.5090,  1.1630],
    [ 0.0000, -1.0180, -1.1630], [ 0.8817,  0.5090, -1.1630], [-0.8817,  0.5090, -1.1630]])

In [ ]:
# --- water, PyFock
water = water_start.copy()
water.calc = PyFockCalculator(functional='skala-1.1', basis='def2-SVP',
                              auxbasis='def2-universal-jkfit', ncores=NCORES, conv_crit=1e-8,
                              sao=True, save_ao_values=True, dispersion=True,
                              dispersion_kwargs={'xc': 'b3lyp5'}, directory='opt_water_pyfock')
t0 = time.perf_counter()
LBFGSLineSearch(water, logfile=None).run(fmax=0.02)
water_pyfock_time = time.perf_counter() - t0
water_pyfock_oh = water.get_distance(0, 1)

print('PyFock water   O-H %.4f A   %.1f s' % (water_pyfock_oh, water_pyfock_time))

In [ ]:
# --- water, PySCF
water = water_start.copy()
water.calc = Skala(xc='skala-1.1', basis='def2-SVP', with_density_fit=True,
                   auxbasis='def2-universal-jkfit')
t0 = time.perf_counter()
LBFGSLineSearch(water, logfile=None).run(fmax=0.02)
water_pyscf_time = time.perf_counter() - t0
water_pyscf_oh = water.get_distance(0, 1)

print('PySCF  water   O-H %.4f A   %.1f s' % (water_pyscf_oh, water_pyscf_time))

In [ ]:
# --- ethane, PyFock
ethane = ethane_start.copy()
ethane.calc = PyFockCalculator(functional='skala-1.1', basis='def2-SVP',
                               auxbasis='def2-universal-jkfit', ncores=NCORES, conv_crit=1e-8,
                               sao=True, save_ao_values=True, dispersion=True,
                               dispersion_kwargs={'xc': 'b3lyp5'}, directory='opt_ethane_pyfock')
t0 = time.perf_counter()
LBFGSLineSearch(ethane, logfile=None).run(fmax=0.02)
ethane_pyfock_time = time.perf_counter() - t0
ethane_pyfock_cc = ethane.get_distance(0, 1)

print('PyFock ethane  C-C %.4f A   %.1f s' % (ethane_pyfock_cc, ethane_pyfock_time))

In [ ]:
# --- ethane, PySCF
ethane = ethane_start.copy()
ethane.calc = Skala(xc='skala-1.1', basis='def2-SVP', with_density_fit=True,
                    auxbasis='def2-universal-jkfit')
t0 = time.perf_counter()
LBFGSLineSearch(ethane, logfile=None).run(fmax=0.02)
ethane_pyscf_time = time.perf_counter() - t0
ethane_pyscf_cc = ethane.get_distance(0, 1)

print('PySCF  ethane  C-C %.4f A   %.1f s' % (ethane_pyscf_cc, ethane_pyscf_time))

In [ ]:
print('water   O-H   PyFock %.4f A   PySCF %.4f A   diff %.1f mA' % (
    water_pyfock_oh, water_pyscf_oh, abs(water_pyfock_oh - water_pyscf_oh) * 1000))
print('ethane  C-C   PyFock %.4f A   PySCF %.4f A   diff %.1f mA' % (
    ethane_pyfock_cc, ethane_pyscf_cc, abs(ethane_pyfock_cc - ethane_pyscf_cc) * 1000))
print()
print('water   PyFock %.1f s   PySCF %.1f s   speed-up %.2fx' % (
    water_pyfock_time, water_pyscf_time, water_pyscf_time / water_pyfock_time))
print('ethane  PyFock %.1f s   PySCF %.1f s   speed-up %.2fx' % (
    ethane_pyfock_time, ethane_pyscf_time, ethane_pyscf_time / ethane_pyfock_time))

## Notes

* `dft.sao = True` matters. PyFock defaults to Cartesian AOs and PySCF to spherical; without it the two
  are not solving the same problem.
* The warm-up is not optional — PyFock's first gradient compiles the AO-Hessian kernel, which can
  dominate a small calculation entirely.
* Both sides apply D3: PyFock through `dispersion=True`, the official API through its `with_dftd3=True`
  default.
* `save_ao_values = True` caches the AO values and gradients on the grid across SCF iterations, which is
  worth about 16% here (65.8 s -> 55.2 s on benzene, identical energy). It costs roughly
  `4 * nao * ngrid * 8` bytes -- 0.4 GB for benzene/def2-SVP -- so drop it if memory is tight.